# Detect why there is pneumonia

Script to detect brightness in an image.

In [ ]:
import os
from PIL import Image
import numpy as np

def find_mean_brightness(folder_path):
    brightness_values = []
    std_values = []

    for root, _, filenames in os.walk(folder_path):
        for filename in filenames:
            if not filename.lower().endswith((".png", ".jpg", ".jpeg")):
                continue

            image_path = os.path.join(root, filename)
            with Image.open(image_path) as image:
                img_l = image.convert("L")
                pixels_array = np.array(img_l, dtype=np.float32)

                image_std = float(np.std(pixels_array))
                std_values.append(image_std)

                image_mean_brightness = float(np.mean(pixels_array))
                brightness_values.append(image_mean_brightness)

    if not brightness_values:
        return 0.0, 0.0, 0.0, 0

    mean_brightness = float(np.mean(brightness_values))
    mean_brightness_percent = (mean_brightness / 255.0) * 100.0
    mean_std = float(np.mean(std_values))
    return mean_brightness, mean_brightness_percent, mean_std, len(brightness_values)

In [ ]:
def __main__():
    print("Check normal patient")
    dataset_folder_normal_patient = os.path.abspath("../output/chest_xray/test/NORMAL")
    print(f"Calculating mean brightness from: {dataset_folder_normal_patient}")
    normal_mean, normal_percent, normal_std, normal_count = find_mean_brightness(dataset_folder_normal_patient)

    print("Check pneumonia patient")
    dataset_folder_pneumonia_patient = os.path.abspath("../output/chest_xray/test/PNEUMONIA")
    print(f"Calculating mean brightness from: {dataset_folder_pneumonia_patient}")
    pneumonia_mean, pneumonia_percent, pneumonia_std, pneumonia_count = find_mean_brightness(dataset_folder_pneumonia_patient)

    print(f"Normal images: {normal_count}")
    print(f"Pneumonia images: {pneumonia_count}")
    print(f"Mean brightness normal patient: {normal_mean:.2f}/255 ({normal_percent:.2f}%)")
    print(f"Mean brightness pneumonia patient: {pneumonia_mean:.2f}/255 ({pneumonia_percent:.2f}%)")
    print(f"Mean std normal patient: {normal_std:.2f}")
    print(f"Mean std pneumonia patient: {pneumonia_std:.2f}")

__main__()

Heatmap of the most impacted areas

In [ ]:
import matplotlib as plt

def generate_heatmap(folder_path):
    grid_size = 32
    target_size = 256
    step = target_size // grid_size
    
    combined_heatmap = np.zeros((grid_size, grid_size), dtype=np.float32)
    count = 0
    print(f"Grid size: {grid_size}x{grid_size}, Step: {step}")
    for root, _, filenames in os.walk(folder_path):
        for filename in filenames:
            if filename.endswith((".png", ".jpg", ".jpeg")):
                image_path = os.path.join(root, filename)
                with Image.open(image_path) as image:
                    img_l = image.convert("L")
                    pixels_array = np.array(img_l, dtype=np.float32)
                    current_heatmap = np.zeros((grid_size, grid_size))
            for i in range (grid_size):
                for j in range(grid_size):
                    block = pixels_array[i*step:(i+1)*step, j*step:(j+1)*step]
                    current_heatmap[i,j] = np.mean(block)
            combined_heatmap += current_heatmap
            count += 1
    average_heatmap = combined_heatmap / count
    
    return average_heatmap


Here the output graph defines this:

The normal signature in this case shows a much clearer image because there is no inflammation or fluid, which means it is filled with air and therefore black on X-rays, a sign of good health.
The signature of a sick person has much more "white" in the sense that the more "white" the image has, the blurrier it is, the more risk there is of inflammation and therefore fluid, which makes the image unclear.

The third image allows us to see where the lungs are and therefore see a concentration of white, often a sign of where the infection is developing.
Red = sick person
Blue = healthy person

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

def main():
    path_normal_patient = os.path.abspath("../output/chest_xray/test/NORMAL")
    path_pneumonia_patient = os.path.abspath("../output/chest_xray/test/PNEUMONIA")
    
    
    print("Generating heatmaps for normal patient")
    avg_heat_normal = generate_heatmap(path_normal_patient)

    print("Generating heatmaps for pneumonia patient")
    avg_heat_pneumonia = generate_heatmap(path_pneumonia_patient)
    
    if avg_heat_normal is not None and avg_heat_pneumonia is not None:
        diff = ((avg_heat_pneumonia - avg_heat_normal) / avg_heat_normal) * 100
        plt.figure(figsize=(15, 5))

        plt.subplot(1, 3, 1)
        plt.imshow(avg_heat_normal, cmap='hot')
        plt.title("Normal")

        plt.subplot(1, 3, 2)
        plt.imshow(avg_heat_pneumonia, cmap='hot')
        plt.title("Pneumonia")

        plt.subplot(1, 3, 3)
        plt.imshow(diff, cmap='RdBu_r', vmin=-20, vmax=20)
        plt.colorbar(label='% Difference')
        plt.title("Correlation Zones (%)")

        plt.show()
        
if __name__ == "__main__":
    main()

Test using a RandomForestClassifier on the train set

In [ ]:
import numpy as np
import os
from PIL import Image
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score, confusion_matrix, r2_score
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

def get_center_image(arr, npix):
    center = [shape // 2 for shape in arr.shape]
    slices = tuple(slice(max(0, c - npix), c + npix) for c in center)
    return arr[slices]

def load_images_from_folder(folder, label, npix=50):
    images = []
    labels = []
    
    if not os.path.exists(folder):
        print(f"ERROR: Directory not found -> {os.path.abspath(folder)}")
        return images, labels

    print(f"Loading images from {folder}...")
    for filename in os.listdir(folder):
        path = os.path.join(folder, filename)
        if path.lower().endswith(('.png', '.jpg', '.jpeg')):
            try:
                img = Image.open(path).convert('L')
                arr = np.array(img)
                
                arr_centered = get_center_image(arr, npix)
                
                images.append(arr_centered.flatten())
                labels.append(label)
            except Exception as e:
                print(f"Skipped {path}: {e}")
                
    return images, labels

path_normal = "../output/chest_xray/train/NORMAL"
path_pneu = "../output/chest_xray/train/PNEUMONIA"

train_normal_x, train_normal_y = load_images_from_folder(path_normal, 0)
train_pneu_x, train_pneu_y = load_images_from_folder(path_pneu, 1)

print(f"Successfully loaded {len(train_normal_x)} normal images and {len(train_pneu_x)} pneumonia images.")

if len(train_normal_x) == 0 or len(train_pneu_x) == 0:
    raise ValueError("directory is empty or not found. Please check your paths.")

X = np.array(train_normal_x + train_pneu_x)
y = np.array(train_normal_y + train_pneu_y)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=True)
print(f"Train set: {len(X_train)} samples, Test set: {len(X_test)} samples")

model = RandomForestClassifier(n_estimators=100, random_state=42)
print("Training model...")
model.fit(X_train, y_train)
print("Model trained successfully.")

print("Evaluating model...")
y_proba = model.predict_proba(X_test)[:, 1]
threshold = 0.65
y_pred_strict = (y_proba >= threshold)

print("\nRESULTS\n")
print(f"Accuracy: {accuracy_score(y_test, y_pred_strict):.4f}")
print("Classification Report:\n", classification_report(y_test, y_pred_strict))

if y_proba is not None:
    roc_auc = roc_auc_score(y_test, y_proba)
    r2 = r2_score(y_test, y_proba) 
    print(f"ROC AUC: {roc_auc:.4f}")
    print(f"R2 Score (on probabilities): {r2:.4f}")
else:
    print("Probabilities unavailable for ROC AUC and R2.")
        
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_strict))

if y_proba is not None:
    print("Generating plot...")
    plt.figure(figsize=(8, 6))
    
    jitter = np.random.normal(0, 0.05, size=len(y_test))
    plt.scatter(y_test + jitter, y_proba, alpha=0.5, color='royalblue')
    
    plt.axhline(threshold, color='red', linestyle='--', label='Decision Threshold ({threshold})')
    
    plt.xlabel('True Label (jittered for visibility)')
    plt.ylabel('Predicted Probability of Pneumonia')
    plt.title(f'Random Forest Predictions (ROC AUC = {roc_auc:.4f} | R2 = {r2:.4f})')
    plt.xticks([0, 1], ['Normal (0)', 'Pneumonia (1)'])
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

Test using a RandomForestClassifier on the test set

In [ ]:
import numpy as np
import os
from PIL import Image
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score, confusion_matrix, r2_score
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

def get_center_image(arr, npix):
    center = [shape // 2 for shape in arr.shape]
    slices = tuple(slice(max(0, c - npix), c + npix) for c in center)
    return arr[slices]

def load_images_from_folder(folder, label, npix=50):
    images = []
    labels = []
    
    if not os.path.exists(folder):
        print(f"Directory not found")
        return images, labels

    print(f"Loading images from {folder}...")
    for filename in os.listdir(folder):
        path = os.path.join(folder, filename)
        if path.lower().endswith(('.png', '.jpg', '.jpeg')):
            try:
                img = Image.open(path).convert('L')
                arr = np.array(img)
                
                arr_centered = get_center_image(arr, npix)
                
                images.append(arr_centered.flatten())
                labels.append(label)
            except Exception as e:
                print(f"Skipped {path}: {e}")
                
    return images, labels

path_normal = "../output/chest_xray/test/NORMAL"
path_pneu = "../output/chest_xray/test/PNEUMONIA"

train_normal_x, train_normal_y = load_images_from_folder(path_normal, 0)
train_pneu_x, train_pneu_y = load_images_from_folder(path_pneu, 1)

print(f"Successfully loaded {len(train_normal_x)} normal images and {len(train_pneu_x)} pneumonia images.")

if len(train_normal_x) == 0 or len(train_pneu_x) == 0:
    raise ValueError("directory is empty or not found. Please check your paths.")

X = np.array(train_normal_x + train_pneu_x)
y = np.array(train_normal_y + train_pneu_y)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=True)
print(f"Train set: {len(X_train)} samples, Test set: {len(X_test)} samples")

model = RandomForestClassifier(n_estimators=100, random_state=42)
print("Training model...")
model.fit(X_train, y_train)
print("Model trained successfully.")

print("Evaluating model...")
y_proba = model.predict_proba(X_test)[:, 1]
threshold = 0.65
y_pred_strict = (y_proba >= threshold)

print("\nRESULTS\n")
print(f"Accuracy: {accuracy_score(y_test, y_pred_strict):.4f}")
print("Classification Report:\n", classification_report(y_test, y_pred_strict))

if y_proba is not None:
    roc_auc = roc_auc_score(y_test, y_proba)
    r2 = r2_score(y_test, y_proba) 
    print(f"ROC AUC: {roc_auc:.4f}")
    print(f"R2 Score (on probabilities): {r2:.4f}")
else:
    print("Probabilities unavailable for ROC AUC and R2.")
        
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_strict))

if y_proba is not None:
    print("Generating plot...")
    plt.figure(figsize=(8, 6))
    
    jitter = np.random.normal(0, 0.05, size=len(y_test))
    plt.scatter(y_test + jitter, y_proba, alpha=0.5, color='royalblue')
    
    plt.axhline(threshold, color='red', linestyle='--', label='Decision Threshold ({threshold})')
    
    plt.xlabel('True Label (jittered for visibility)')
    plt.ylabel('Predicted Probability of Pneumonia')
    plt.title(f'Random Forest Predictions (ROC AUC = {roc_auc:.4f} | R2 = {r2:.4f})')
    plt.xticks([0, 1], ['Normal (0)', 'Pneumonia (1)'])
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

AI training on the **TRAIN** dataset to then verify on the **TEST** dataset

In [ ]:
import numpy as np
import os
from PIL import Image
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score, confusion_matrix, r2_score
import matplotlib.pyplot as plt

def get_center_image(arr, npix):
    center = [shape // 2 for shape in arr.shape]
    slices = tuple(slice(max(0, c - npix), c + npix) for c in center)
    return arr[slices]

def load_images_from_folder(folder, label, npix=50):
    images = []
    labels = []
    
    if not os.path.exists(folder):
        print(f"Directory not found")
        return images, labels

    print(f"Loading images from {folder}...")
    for filename in os.listdir(folder):
        path = os.path.join(folder, filename)
        if path.lower().endswith(('.png', '.jpg', '.jpeg')):
            try:
                img = Image.open(path).convert('L')
                arr = np.array(img)
                
                arr_centered = get_center_image(arr, npix)
                
                images.append(arr_centered.flatten())
                labels.append(label)
            except Exception as e:
                print(f"Skipped {path}: {e}")
                
    return images, labels

train_normal_dir = "../output/chest_xray/train/NORMAL"
train_pneu_dir = "../output/chest_xray/train/PNEUMONIA"

test_normal_dir = "../output/chest_xray/test/NORMAL"
test_pneu_dir = "../output/chest_xray/test/PNEUMONIA"

train_normal_x, train_normal_y = load_images_from_folder(train_normal_dir, 0)
train_pneu_x, train_pneu_y = load_images_from_folder(train_pneu_dir, 1)

if len(train_normal_x) == 0 or len(train_pneu_x) == 0:
    raise ValueError("Train directories are empty or not found.")

X_train = np.array(train_normal_x + train_pneu_x)
y_train = np.array(train_normal_y + train_pneu_y)
print(f"\nTrain set loaded: {len(X_train)} samples.")

test_normal_x, test_normal_y = load_images_from_folder(test_normal_dir, 0)
test_pneu_x, test_pneu_y = load_images_from_folder(test_pneu_dir, 1)

if len(test_normal_x) == 0 or len(test_pneu_x) == 0:
    raise ValueError("Test directories are empty or not found.")

X_test = np.array(test_normal_x + test_pneu_x)
y_test = np.array(test_normal_y + test_pneu_y)
print(f"Test set loaded: {len(X_test)} samples.\n")

model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight="balanced")
print("Training model on the FULL train set...")
model.fit(X_train, y_train)
print("Model trained successfully.\n")

print("Evaluating model on the test set...")
y_proba = model.predict_proba(X_test)[:, 1]

threshold = 0.65
y_pred_strict = (y_proba >= threshold).astype(int)

print("\nRESULTS\n")
print(f"Accuracy: {accuracy_score(y_test, y_pred_strict):.4f}")
print("Classification Report:\n", classification_report(y_test, y_pred_strict))

if y_proba is not None:
    roc_auc = roc_auc_score(y_test, y_proba)
    r2 = r2_score(y_test, y_proba) 
    print(f"ROC AUC: {roc_auc:.4f}")
    print(f"R2 Score (on probabilities): {r2:.4f}")
        
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_strict))

if y_proba is not None:
    print("Generating plot...")
    plt.figure(figsize=(8, 6))
    
    jitter = np.random.normal(0, 0.05, size=len(y_test))
    plt.scatter(y_test + jitter, y_proba, alpha=0.5, color='royalblue')
    
    plt.axhline(threshold, color='red', linestyle='--', label=f'Decision Threshold ({threshold})')
    
    plt.xlabel('True Label (jittered for visibility)')
    plt.ylabel('Predicted Probability of Pneumonia')
    plt.title(f'Random Forest Predictions (ROC AUC = {roc_auc:.4f})')
    plt.xticks([0, 1], ['Normal (0)', 'Pneumonia (1)'])
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

Pneumonia detection based on 3 visual characteristics:

Mean density: Measures the overall amount of "white" in the lungs (sick lungs are more opaque).
Heterogeneity (Standard deviation): Measures the strong contrasts between white and black areas, which reveals spots of infection.
Asymmetry: Compares the left and right lung to see if the image is unbalanced (infection often affects only one side).

In [ ]:
import numpy as np
import os
import pandas as pd
from PIL import Image
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

def extract_clinical_features(arr, npix=50):
    center = [shape // 2 for shape in arr.shape]
    slices = tuple(slice(max(0, c - npix), c + npix) for c in center)
    cropped = arr[slices]
    
    density = np.mean(cropped)
    
    heterogeneity = np.std(cropped)
    
    width = cropped.shape[1]
    left_lung = cropped[:, :width//2]
    right_lung = cropped[:, width//2:]
    asymmetry = abs(np.mean(left_lung) - np.mean(right_lung))
    
    return [density, heterogeneity, asymmetry]

def load_extracted_features(folder, label, npix=50):
    features_list = []
    labels = []
    
    if not os.path.exists(folder):
        print(f"ERROR: Directory not found -> {os.path.abspath(folder)}")
        return features_list, labels

    print(f"Extracting features from {folder}...")
    for filename in os.listdir(folder):
        path = os.path.join(folder, filename)
        if path.lower().endswith(('.png', '.jpg', '.jpeg')):
            try:
                img = Image.open(path).convert('L')
                arr = np.array(img)
                
                features = extract_clinical_features(arr, npix)
                
                features_list.append(features)
                labels.append(label)
            except Exception as e:
                print(f"Skipped {path}: {e}")
                
    return features_list, labels

train_normal_dir = "../output/chest_xray/train/NORMAL"
train_pneu_dir = "../output/chest_xray/train/PNEUMONIA"
test_normal_dir = "../output/chest_xray/test/NORMAL"
test_pneu_dir = "../output/chest_xray/test/PNEUMONIA"

train_normal_x, train_normal_y = load_extracted_features(train_normal_dir, 0)
train_pneu_x, train_pneu_y = load_extracted_features(train_pneu_dir, 1)

test_normal_x, test_normal_y = load_extracted_features(test_normal_dir, 0)
test_pneu_x, test_pneu_y = load_extracted_features(test_pneu_dir, 1)

X_train = np.array(train_normal_x + train_pneu_x)
y_train = np.array(train_normal_y + train_pneu_y)
X_test = np.array(test_normal_x + test_pneu_x)
y_test = np.array(test_normal_y + test_pneu_y)

print(f"\nExtraction complete: {len(X_train)} train samples, {len(X_test)} test samples.")

model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight="balanced", max_depth=5)
print("Training interpretable model...")
model.fit(X_train, y_train)

y_proba = model.predict_proba(X_test)[:, 1]
threshold = 0.55
y_pred_strict = (y_proba >= threshold).astype(int)

roc_auc = roc_auc_score(y_test, y_proba)
print("\n--- Evaluation Results ---")
print(f"ROC AUC: {roc_auc:.4f}")
print(f"Accuracy: {accuracy_score(y_test, y_pred_strict):.4f}")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_strict))

feature_names = ["Mean Density", "Heterogeneity (Std)", "Left/Right Asymmetry"]
importances = model.feature_importances_

plt.figure(figsize=(8, 5))
sns.barplot(x=importances, y=feature_names, palette="viridis", hue=feature_names, legend=False)
plt.title("Importance of Clinical Signs for AI (Explainability)")
plt.xlabel("Weight in decision (Percentage of importance)")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

Detection of whether a patient has pneumonia using visual characteristics

train in the train forlder and test in the from folder

Pneumonia detection based on 3 visual characteristics:

Mean density: Measures the overall amount of "white" in the lungs (sick lungs are more opaque).
Heterogeneity (Standard deviation): Measures the strong contrasts between white and black areas, which reveals spots of infection.
Asymmetry: Compares the left and right lung to see if the image is unbalanced (infection often affects only one side).

In [ ]:
import os
import numpy as np
from PIL import Image
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score, confusion_matrix, r2_score
from sklearn.tree import export_text
import matplotlib.pyplot as plt

def extract_clinical_features(arr, npix):
    center = [shape // 2 for shape in arr.shape]
    slices = tuple(slice(max(0, c - npix), c + npix) for c in center)
    cropped = arr[slices]
    
    density = np.mean(cropped)
    
    heterogeneity = np.std(cropped)
    
    width = cropped.shape[1]
    left_lung = cropped[:, :width//2]
    right_lung = cropped[:, width//2:]
    asymmetry = abs(np.mean(left_lung) - np.mean(right_lung))
    
    return [density, heterogeneity, asymmetry]

def load_images_from_folder(folder, label, npix=50):
    features_list = []
    labels = []
    
    if not os.path.exists(folder):
        print(f"Directory not found: {folder}")
        return features_list, labels

    print(f"Extracting clinical features from {folder}...")
    for filename in os.listdir(folder):
        path = os.path.join(folder, filename)
        if path.lower().endswith(('.png', '.jpg', '.jpeg')):
            try:
                img = Image.open(path).convert('L')
                arr = np.array(img)
                
                features = extract_clinical_features(arr, npix)
                
                features_list.append(features)
                labels.append(label)
            except Exception as e:
                print(f"Skipped {path}: {e}")
                
    return features_list, labels

train_normal_dir = "../output/chest_xray/train/NORMAL"
train_pneu_dir = "../output/chest_xray/train/PNEUMONIA"
test_normal_dir = "../output/chest_xray/test/NORMAL"
test_pneu_dir = "../output/chest_xray/test/PNEUMONIA"

train_normal_x, train_normal_y = load_images_from_folder(train_normal_dir, 0)
train_pneu_x, train_pneu_y = load_images_from_folder(train_pneu_dir, 1)

if len(train_normal_x) == 0 or len(train_pneu_x) == 0:
    raise ValueError("Train directories are empty or not found.")

X_train = np.array(train_normal_x + train_pneu_x)
y_train = np.array(train_normal_y + train_pneu_y)
print(f"\nTrain set loaded: {len(X_train)} samples.")

test_normal_x, test_normal_y = load_images_from_folder(test_normal_dir, 0)
test_pneu_x, test_pneu_y = load_images_from_folder(test_pneu_dir, 1)

if len(test_normal_x) == 0 or len(test_pneu_x) == 0:
    raise ValueError("Test directories are empty or not found.")

X_test = np.array(test_normal_x + test_pneu_x)
y_test = np.array(test_normal_y + test_pneu_y)
print(f"Test set loaded: {len(X_test)} samples.\n")

model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight="balanced", max_depth=5)
print("Training INTERPRETABLE model on the FULL train set...")
model.fit(X_train, y_train)
print("Model trained successfully.\n")

print("Evaluating model on the test set...")
y_proba = model.predict_proba(X_test)[:, 1]

threshold = 0.55 
y_pred_strict = (y_proba >= threshold).astype(int)

print("\n--- RESULTS ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred_strict):.4f}")
print("Classification Report:\n", classification_report(y_test, y_pred_strict))

if y_proba is not None:
    roc_auc = roc_auc_score(y_test, y_proba)
    print(f"ROC AUC: {roc_auc:.4f}")
        
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_strict))

feature_names = ["Density (Mean)", "Heterogeneity (Std)", "Asymmetry"]

print("\n--- DECISION RULES (Sample from one tree) ---")
tree_rules = export_text(model.estimators_[0], feature_names=feature_names)
print(tree_rules)

if y_proba is not None:
    print("Generating plots...")
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    jitter = np.random.normal(0, 0.05, size=len(y_test))
    ax1.scatter(y_test + jitter, y_proba, alpha=0.5, color='royalblue')
    ax1.axhline(threshold, color='red', linestyle='--', label=f'Decision Threshold ({threshold})')
    ax1.set_xlabel('True Label (jittered for visibility)')
    ax1.set_ylabel('Predicted Probability of Pneumonia')
    ax1.set_title(f'Predictions (ROC AUC = {roc_auc:.4f})')
    ax1.set_xticks([0, 1])
    ax1.set_xticklabels(['Normal (0)', 'Pneumonia (1)'])
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    importances = model.feature_importances_
    ax2.barh(feature_names, importances, color=['#475b82', '#2e857e', '#6ebe6a'])
    ax2.set_xlabel('Weight in Decision (%)')
    ax2.set_title('Clinical Features Importance')
    
    plt.tight_layout()
    plt.show()

SAME CODE BUT WITH A GRIDSEARCH for find best parameters

In [ ]:
import os
import numpy as np
from PIL import Image
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score, confusion_matrix
from sklearn.tree import export_text
from sklearn.model_selection import GridSearchCV # L'outil magique pour trouver les meilleurs paramètres
import matplotlib.pyplot as plt

def extract_clinical_features(arr, npix):
    center = [shape // 2 for shape in arr.shape]
    slices = tuple(slice(max(0, c - npix), c + npix) for c in center)
    cropped = arr[slices]
    
    density = np.mean(cropped)
    heterogeneity = np.std(cropped)
    
    width = cropped.shape[1]
    left_lung = cropped[:, :width//2]
    right_lung = cropped[:, width//2:]
    asymmetry = abs(np.mean(left_lung) - np.mean(right_lung))
    
    return [density, heterogeneity, asymmetry]

def load_images_from_folder(folder, label, npix=50):
    features_list = []
    labels = []
    
    if not os.path.exists(folder):
        print(f"Directory not found: {folder}")
        return features_list, labels

    print(f"Extracting clinical features from {folder}...")
    for filename in os.listdir(folder):
        path = os.path.join(folder, filename)
        if path.lower().endswith(('.png', '.jpg', '.jpeg')):
            try:
                img = Image.open(path).convert('L')
                arr = np.array(img)
                
                features = extract_clinical_features(arr, npix)
                features_list.append(features)
                labels.append(label)
            except Exception as e:
                print(f"Skipped {path}: {e}")
                
    return features_list, labels

# --- 1. Chargement des données ---
train_normal_dir = "../output/chest_xray/train/NORMAL"
train_pneu_dir = "../output/chest_xray/train/PNEUMONIA"
test_normal_dir = "../output/chest_xray/test/NORMAL"
test_pneu_dir = "../output/chest_xray/test/PNEUMONIA"

train_normal_x, train_normal_y = load_images_from_folder(train_normal_dir, 0)
train_pneu_x, train_pneu_y = load_images_from_folder(train_pneu_dir, 1)

if len(train_normal_x) == 0 or len(train_pneu_x) == 0:
    raise ValueError("Train directories are empty or not found.")

X_train = np.array(train_normal_x + train_pneu_x)
y_train = np.array(train_normal_y + train_pneu_y)
print(f"\nTrain set loaded: {len(X_train)} samples.")

test_normal_x, test_normal_y = load_images_from_folder(test_normal_dir, 0)
test_pneu_x, test_pneu_y = load_images_from_folder(test_pneu_dir, 1)

if len(test_normal_x) == 0 or len(test_pneu_x) == 0:
    raise ValueError("Test directories are empty or not found.")

X_test = np.array(test_normal_x + test_pneu_x)
y_test = np.array(test_normal_y + test_pneu_y)
print(f"Test set loaded: {len(X_test)} samples.\n")

# --- 2. Configuration de GridSearchCV ---
print("Setting up GridSearchCV...")
# On crée un modèle de base
base_model = RandomForestClassifier(random_state=42, class_weight="balanced")

# On définit la grille des paramètres à tester
# Le modèle va tester toutes les combinaisons possibles entre ces valeurs
param_grid = {
    'n_estimators': [50, 100, 200, 500],       # Le nombre d'arbres
    'max_depth': [3, 5, 7, 10, 25 ,50],           # La profondeur maximale (plus c'est profond, plus c'est complexe)
    'min_samples_leaf': [1, 5, 10, 25, 50]        # Nombre minimum d'échantillons pour valider une feuille de l'arbre
}

# cv=5 signifie qu'il va faire de la validation croisée (découper le train set en 5 pour s'auto-évaluer)
# n_jobs=-1 permet d'utiliser tous les cœurs de votre processeur pour aller plus vite
grid_search = GridSearchCV(estimator=base_model, param_grid=param_grid, 
                           cv=5, scoring='accuracy', n_jobs=-1, verbose=1)

# --- 3. Entraînement et Recherche ---
print("Training and tuning model on the FULL train set (this may take a moment)...")
grid_search.fit(X_train, y_train)

# --- 4. Récupération du meilleur modèle ---
print("\n--- GRID SEARCH RESULTS ---")
print(f"Best parameters found: {grid_search.best_params_}")
model = grid_search.best_estimator_ # <-- C'est notre nouveau modèle champion !

# --- 5. Évaluation du modèle champion sur le Test Set ---
print("\nEvaluating BEST model on the test set...")
y_proba = model.predict_proba(X_test)[:, 1]

threshold = 0.67
y_pred_strict = (y_proba >= threshold).astype(int)

print("\n--- RESULTS ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred_strict):.4f}")
print("Classification Report:\n", classification_report(y_test, y_pred_strict))

if y_proba is not None:
    roc_auc = roc_auc_score(y_test, y_proba)
    print(f"ROC AUC: {roc_auc:.4f}")
        
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_strict))

feature_names = ["Density (Mean)", "Heterogeneity (Std)", "Asymmetry"]

print("\n--- DECISION RULES (Sample from one tree) ---")
# On imprime les règles du premier arbre du modèle champion
tree_rules = export_text(model.estimators_[0], feature_names=feature_names)
print(tree_rules)

# --- 6. Affichage des graphiques ---
if y_proba is not None:
    print("Generating plots...")
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    jitter = np.random.normal(0, 0.05, size=len(y_test))
    ax1.scatter(y_test + jitter, y_proba, alpha=0.5, color='royalblue')
    ax1.axhline(threshold, color='red', linestyle='--', label=f'Decision Threshold ({threshold})')
    ax1.set_xlabel('True Label (jittered for visibility)')
    ax1.set_ylabel('Predicted Probability of Pneumonia')
    ax1.set_title(f'Predictions (ROC AUC = {roc_auc:.4f})')
    ax1.set_xticks([0, 1])
    ax1.set_xticklabels(['Normal (0)', 'Pneumonia (1)'])
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    importances = model.feature_importances_
    ax2.barh(feature_names, importances, color=['#475b82', '#2e857e', '#6ebe6a'])
    ax2.set_xlabel('Weight in Decision (%)')
    ax2.set_title('Clinical Features Importance')
    
    plt.tight_layout()
    plt.show()

In [ ]:
import os
import numpy as np
from PIL import Image
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score, confusion_matrix
from sklearn.tree import export_text
import matplotlib.pyplot as plt

def extract_clinical_features(arr, npix):
    center = [shape // 2 for shape in arr.shape]
    slices = tuple(slice(max(0, c - npix), c + npix) for c in center)
    cropped = arr[slices]
    
    h, w = cropped.shape
    half_h, half_w = h // 2, w // 2
    
    tl = cropped[:half_h, :half_w]
    tr = cropped[:half_h, half_w:]
    bl = cropped[half_h:, :half_w]
    br = cropped[half_h:, half_w:]
    
    features = [
        np.mean(tl), np.std(tl),
        np.mean(tr), np.std(tr),
        np.mean(bl), np.std(bl),
        np.mean(br), np.std(br)
    ]
    
    return features

def load_images_from_folder(folder, label, npix=50):
    features_list = []
    labels = []
    
    if not os.path.exists(folder):
        print(f"Directory not found: {folder}")
        return features_list, labels

    print(f"Extracting clinical features from {folder}...")
    for filename in os.listdir(folder):
        path = os.path.join(folder, filename)
        if path.lower().endswith(('.png', '.jpg', '.jpeg')):
            try:
                img = Image.open(path).convert('L')
                arr = np.array(img)
                
                features = extract_clinical_features(arr, npix)
                features_list.append(features)
                labels.append(label)
            except Exception as e:
                print(f"Skipped {path}: {e}")
                
    return features_list, labels

train_normal_dir = "../output/chest_xray/train/NORMAL"
train_pneu_dir = "../output/chest_xray/train/PNEUMONIA"
test_normal_dir = "../output/chest_xray/test/NORMAL"
test_pneu_dir = "../output/chest_xray/test/PNEUMONIA"

train_normal_x, train_normal_y = load_images_from_folder(train_normal_dir, 0)
train_pneu_x, train_pneu_y = load_images_from_folder(train_pneu_dir, 1)

if len(train_normal_x) == 0 or len(train_pneu_x) == 0:
    raise ValueError("Train directories are empty or not found.")

X_train = np.array(train_normal_x + train_pneu_x)
y_train = np.array(train_normal_y + train_pneu_y)
print(f"\nTrain set loaded: {len(X_train)} samples.")

test_normal_x, test_normal_y = load_images_from_folder(test_normal_dir, 0)
test_pneu_x, test_pneu_y = load_images_from_folder(test_pneu_dir, 1)

if len(test_normal_x) == 0 or len(test_pneu_x) == 0:
    raise ValueError("Test directories are empty or not found.")

X_test = np.array(test_normal_x + test_pneu_x)
y_test = np.array(test_normal_y + test_pneu_y)
print(f"Test set loaded: {len(X_test)} samples.\n")

model = RandomForestClassifier(
    n_estimators=200, 
    max_depth=50, 
    min_samples_leaf=1, 
    random_state=42, 
    class_weight="balanced"
)

print("Training model on the FULL train set...")
model.fit(X_train, y_train)
print("Model trained successfully.\n")

print("Evaluating model on the test set...")
y_proba = model.predict_proba(X_test)[:, 1]

threshold = 0.67
y_pred_strict = (y_proba >= threshold).astype(int)

print("\n--- RESULTS ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred_strict):.4f}")
print("Classification Report:\n", classification_report(y_test, y_pred_strict))

if y_proba is not None:
    roc_auc = roc_auc_score(y_test, y_proba)
    print(f"ROC AUC: {roc_auc:.4f}")
        
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_strict))

feature_names = [
    "TL_Density", "TL_Heterogeneity",
    "TR_Density", "TR_Heterogeneity",
    "BL_Density", "BL_Heterogeneity",
    "BR_Density", "BR_Heterogeneity"
]

print("\n--- DECISION RULES (Sample from one tree) ---")
tree_rules = export_text(model.estimators_[0], feature_names=feature_names)
print(tree_rules)

if y_proba is not None:
    print("Generating plots...")
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    jitter = np.random.normal(0, 0.05, size=len(y_test))
    ax1.scatter(y_test + jitter, y_proba, alpha=0.5, color='royalblue')
    ax1.axhline(threshold, color='red', linestyle='--', label=f'Decision Threshold ({threshold})')
    ax1.set_xlabel('True Label (jittered for visibility)')
    ax1.set_ylabel('Predicted Probability of Pneumonia')
    ax1.set_title(f'Predictions (ROC AUC = {roc_auc:.4f})')
    ax1.set_xticks([0, 1])
    ax1.set_xticklabels(['Normal (0)', 'Pneumonia (1)'])
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    importances = model.feature_importances_
    sorted_idx = np.argsort(importances)
    
    colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(feature_names)))
    
    ax2.barh([feature_names[i] for i in sorted_idx], importances[sorted_idx], color=colors)
    ax2.set_xlabel('Weight in Decision (%)')
    ax2.set_title('Clinical Features Importance (4 Quadrants)')
    
    plt.tight_layout()
    plt.show()

In [ ]:
import os
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns

def extract_clinical_features(arr, npix):
    center = [shape // 2 for shape in arr.shape]
    slices = tuple(slice(max(0, c - npix), c + npix) for c in center)
    cropped = arr[slices]
    
    h, w = cropped.shape
    half_h, half_w = h // 2, w // 2
    
    tl = cropped[:half_h, :half_w]
    tr = cropped[:half_h, half_w:]
    bl = cropped[half_h:, :half_w]
    br = cropped[half_h:, half_w:]
    
    features = [
        np.mean(tl), np.std(tl),
        np.mean(tr), np.std(tr),
        np.mean(bl), np.std(bl),
        np.mean(br), np.std(br)
    ]
    
    return features

def load_images_from_folder(folder, label, npix=50):
    features_list = []
    labels = []
    
    if not os.path.exists(folder):
        print(f"Directory not found: {folder}")
        return features_list, labels

    print(f"Extracting clinical features from {folder}...")
    for filename in os.listdir(folder):
        path = os.path.join(folder, filename)
        if path.lower().endswith(('.png', '.jpg', '.jpeg')):
            try:
                img = Image.open(path).convert('L')
                arr = np.array(img)
                
                features = extract_clinical_features(arr, npix)
                features_list.append(features)
                labels.append(label)
            except Exception as e:
                pass
                
    return features_list, labels

train_normal_dir = "../output/chest_xray/train/NORMAL"
train_pneu_dir = "../output/chest_xray/train/PNEUMONIA"
test_normal_dir = "../output/chest_xray/test/NORMAL"
test_pneu_dir = "../output/chest_xray/test/PNEUMONIA"

train_normal_x, train_normal_y = load_images_from_folder(train_normal_dir, 0)
train_pneu_x, train_pneu_y = load_images_from_folder(train_pneu_dir, 1)

if len(train_normal_x) == 0 or len(train_pneu_x) == 0:
    raise ValueError("Train directories are empty or not found.")

X_train = np.array(train_normal_x + train_pneu_x)
y_train = np.array(train_normal_y + train_pneu_y)

test_normal_x, test_normal_y = load_images_from_folder(test_normal_dir, 0)
test_pneu_x, test_pneu_y = load_images_from_folder(test_pneu_dir, 1)

if len(test_normal_x) == 0 or len(test_pneu_x) == 0:
    raise ValueError("Test directories are empty or not found.")

X_test = np.array(test_normal_x + test_pneu_x)
y_test = np.array(test_normal_y + test_pneu_y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

models = {
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced"),
    "Gradient Boosting": HistGradientBoostingClassifier(random_state=42),
    "Logistic Regression": LogisticRegression(random_state=42, class_weight="balanced", max_iter=1000),
    "Support Vector Machine (SVC)": SVC(probability=True, random_state=42, class_weight="balanced"),
    "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=5)
}

results = []

print("\n--- STARTING ALGORITHM TOURNAMENT ---")

for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train_scaled, y_train)
    
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    
    results.append({"Model": name, "Accuracy": acc, "ROC AUC": auc})

df_results = pd.DataFrame(results).sort_values(by="ROC AUC", ascending=False)

print("\n--- FINAL LEADERBOARD ---")
print(df_results.to_string(index=False))

plt.figure(figsize=(10, 6))
sns.barplot(x="ROC AUC", y="Model", data=df_results, palette="viridis")
plt.title("Algorithm Tournament: ROC AUC Score Comparison")
plt.xlabel("ROC AUC Score")
plt.ylabel("Algorithm")
plt.xlim(0.7, 1.0)
plt.tight_layout()
plt.show()

PROBABLY WRONG, TO DELETE

In [ ]:
import numpy as np
from PIL import Image
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import os

normal_dir = os.path.abspath("../output/chest_xray/train/NORMAL")
pneu_dir = os.path.abspath("../output/chest_xray/train/PNEUMONIA")

def load_multi_features(normal_dir, pneu_dir):
    X = []
    y = []
    for dir_path, label in ((normal_dir, 0), (pneu_dir, 1)):
        if not os.path.exists(dir_path):
            print(f"Dir not found: {dir_path}")
            continue
        for root, _, files in os.walk(dir_path):
            for fname in files:
                if not fname.lower().endswith(('.png', '.jpg', '.jpeg')):
                    continue
                path = os.path.join(root, fname)
                try:
                    with Image.open(path) as img:
                        arr = np.array(img.convert('L'), dtype=np.float32)
                except Exception as e:
                    print(f"Skipped {path}: {e}")
                    continue
                mean_val = float(np.mean(arr))
                std_val = float(np.std(arr))
                min_val = float(np.min(arr))
                max_val = float(np.max(arr))
                median_val = float(np.median(arr))
                X.append([mean_val, std_val, min_val, max_val, median_val])
                y.append(label)
    if not X:
        raise ValueError("No images found in provided folders")
    return np.array(X), np.array(y)

X, y = load_multi_features(normal_dir, pneu_dir)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=True)

model = make_pipeline(StandardScaler(), LinearRegression())
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"LinearRegression MSE: {mse:.4f}")
print(f"LinearRegression R2: {r2:.4f}")

y_pred_class = (y_pred >= 0.5).astype(int)
acc = accuracy_score(y_test, y_pred_class)
print(f"Accuracy (threshold 0.5): {acc:.4f}")
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred_class))

plt.figure(figsize=(8,6))
plt.scatter(y_test + np.random.normal(0, 0.05, size=len(y_test)), y_pred, alpha=0.5)
plt.axhline(0.5, color='r', linestyle='--', label='Threshold 0.5')
plt.xlabel('True Label (jittered for visibility)')
plt.ylabel('Predicted Value (Continuous)')
plt.title(f'LinearRegression with Multiple Features (R2 = {r2:.4f})')
plt.xticks([0, 1], ['Normal (0)', 'Pneumonia (1)'])
plt.legend()
plt.grid(True)
plt.show()

PROBABLY WRONG, TO DELETE